# 🔥 Ember Observatory
### Live scientific training, ForeFire simulations, and distributed resource monitoring

**Modes:** Replay a finished experiment • Monitor Fuzzball jobs live • Inspect real model predictions • Compare bottlenecks.

This notebook reads *actual* histories, events and exported test images. It does **not** invent metrics or start a cluster workflow just by opening it.

**Reading the dashboard**
- **Training / validation:** Updated each epoch. IoU and Dice are segmentation accuracy measures; pixel accuracy may be dominated by unburned background.
- **Test:** Measured only after model selection; intentionally *not* a per-epoch curve.
- **GPU / CPU:** Snapshots are instantaneous, not average utilization. Empty plots mean telemetry was not recorded.
- **DDP fine-tuning:** The timeline shows a transition only if a DDP job truly emits a `ddp_finetune_started` event.
- **ForeFire:** CPU physics runs in separate workers; the surrogate is a learned approximation, not verified wildfire forecasting.


## 1. Setup and locate your experiment artifacts

Run this notebook inside the Ember repo, ideally in VS Code's Jupyter extension. Install display dependencies in your notebook kernel with `python -m pip install jupyter ipykernel matplotlib numpy pandas`. You **do not** need local PyTorch for the dashboard.

Your completed pilot artifacts may be in a separate `Ember-fuzzball-20261114` directory. Change `RUNS_DIR` to match your machine.


In [ ]:
from pathlib import Path
import sys, json
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display, Image, Markdown

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from orchestration.observe import dashboard as obs

# Edit this to your retrieved pilot results directory, if outside the repository.
RUNS_DIR = ROOT / "orchestration" / "fuzzball" / "runs"
print("Repository:", ROOT)
print("Experiment root:", RUNS_DIR, "| exists:", RUNS_DIR.exists())

In [ ]:
histories = sorted(RUNS_DIR.glob("**/history.csv")) if RUNS_DIR.exists() else []
print(f"Found {len(histories)} training histories:")
for i, path in enumerate(histories):
    print(f"  [{i}] {path.relative_to(RUNS_DIR)}")
if not histories:
    print("Set RUNS_DIR to the directory containing retrieved_baseline/pilot/outputs_baseline/history.csv.")

## 2. Real training and validation curves

Select one retrieved run and plot its actual loss, IoU, Dice and reached-cell arrival MAE. Older pilot runs may contain only validation IoU; unavailable metrics are not synthesized. The test results appear in the title as a **single final assessment**.


In [ ]:
RUN_INDEX = 0
if histories:
    history_path = histories[RUN_INDEX]
    results_dir = history_path.parent
    rows = obs.read_history(history_path)
    test_file = results_dir / "test_metrics.json"
    test = json.loads(test_file.read_text()) if test_file.exists() else None
    print("Experiment:", results_dir)
    print("Completed epochs:", len(rows))
    display(obs.plot_learning(rows, test))
    plt.show()
else:
    print("No local histories. Use the Fuzzball live tab below or set RUNS_DIR.")

## 3. Evaluation accuracy and model comparison

Model selection should use **validation loss**. Test scores can be displayed for retrospective reporting, but must never influence which checkpoint is selected. Conditional arrival MAE is computed only on simulator-reached cells: read it alongside IoU and recall.


In [ ]:
summary_rows = []
for path in histories:
    values = obs.read_history(path)
    if not values:
        continue
    best = min(values, key=lambda r: float(r.get("val_loss", "inf")))
    test_path = path.parent / "test_metrics.json"
    test = json.loads(test_path.read_text()) if test_path.exists() else {}
    summary_rows.append({
        "run": path.parent.name,
        "epochs": len(values),
        "best val loss": float(best.get("val_loss", "nan")),
        "best val IoU": float(best.get("val_iou", "nan")),
        "test IoU (report only)": test.get("iou"),
        "test arrival MAE (s)": test.get("arrival_mae_seconds_on_reached"),
    })
if summary_rows:
    display(pd.DataFrame(summary_rows).sort_values("best val loss").style.format(precision=4))
else:
    print("No finished run histories found.")

## 4. CPU/GPU timeline and bottlenecks

The timeline uses observed start/finish events from physics shards, GPU training, optional DDP fine-tuning and final testing. A missing interval means no corresponding event pair was recorded.

Resource plots are **sampled snapshots**; do not call them average GPU utilization.


In [ ]:
event_files = sorted(RUNS_DIR.glob("**/events.jsonl")) if RUNS_DIR.exists() else []
all_events = []
for file in event_files:
    all_events.extend(obs.read_events(file))
print("Structured event files:", len(event_files), "Events:", len(all_events))
print("Bottleneck summary:", obs.bottleneck_report(all_events))
if all_events:
    display(pd.DataFrame(obs.progress_report(all_events)))
if all_events:
    fig = obs.plot_timeline(all_events)
    if fig is not None: display(fig); plt.show()
    fig = obs.plot_resources(all_events)
    if fig is not None: display(fig); plt.show()
else:
    print("Historical pilot outputs predate structured logging. New instrumented runs will show phases here.")

## 5. Live Fuzzball monitoring

Set the real workflow ID and job names from `fuzzball workflow list`. The notebook polls the CLI at a bounded interval. **Stop / Interrupt Kernel** to stop watching.

If your Jupyter kernel runs under Windows, the default command falls back to `wsl.exe --exec /home/caden/.local/bin/fuzzball`. If it runs inside WSL and has the CLI on PATH, it uses `fuzzball` directly.

You can watch the concurrent CPU physics and GPU jobs together. New training runs emit timestamped `EMBER_EVENT` JSON; historical pilot logs fall back to their original epoch text.


In [ ]:
# Replace with the workflow ID printed by YOUR NEW Fuzzball submission.
WORKFLOWS = {
    # "YOUR-ACTUAL-WORKFLOW-UUID": ["train-baseline", "train-lower-lr", "physics-0", "physics-1"]
}
WATCH_INTERVAL_SECONDS = 8
WATCH_CYCLES = 60    # Maximum ~8 minutes; interrupt any time

print("Configured workflows:", WORKFLOWS or "None — add a real workflow ID to monitor.")

In [ ]:
# Run this cell AFTER submitting a new workflow, or leave disabled for replay.
START_LIVE_WATCH = False
if START_LIVE_WATCH and WORKFLOWS:
    obs.watch(WORKFLOWS, interval_s=WATCH_INTERVAL_SECONDS, cycles=WATCH_CYCLES)
else:
    print("Live watch is idle. Set WORKFLOWS and START_LIVE_WATCH=True when a job is running.")

## 6. Real ForeFire versus surrogate image gallery

Images are exported **on the GPU training node**, using real held-out ForeFire numeric targets and the actual trained checkpoint. Displaying them here requires only the retrieved PNGs, not GPU access or local PyTorch.

Colors: actual/estimated arrival times in minutes; mask error uses true-positive, false-positive and false-negative pixels. Plots include IoU/recall at six forecast horizons.


In [ ]:
gallery_pngs = sorted(RUNS_DIR.glob("**/gallery/*.png")) if RUNS_DIR.exists() else []
print(f"Found {len(gallery_pngs)} real prediction gallery images")
for path in gallery_pngs[:4]:
    display(Markdown(f"**{path.stem}** — {path.parent.parent.name}"))
    display(Image(filename=str(path), width=960))
if not gallery_pngs:
    print("Pilot runs made before this branch do not have galleries; instrumented evaluate jobs will export them.")

## 7. Scenario-level accuracy and geographic diagnostics

The pilot validation region (Texas Panhandle) and test region (Montana) have markedly different IoU. Re-evaluate across more independent regions before generalization claims. This section loads actual saved per-case, per-horizon scores from the exported gallery.


In [ ]:
gallery_reports = sorted(RUNS_DIR.glob("**/gallery/gallery_metrics.json")) if RUNS_DIR.exists() else []
scenarios = []
for path in gallery_reports:
    data = json.loads(path.read_text())
    for case in data.get("cases", []):
        for horizon in case.get("horizons", []):
            scenarios.append({"run": path.parent.parent.name,
                              "simulation": case["simulation_id"],
                              "family": case["scenario_family_id"],
                              "horizon_min": horizon["horizon_s"] / 60,
                              "IoU": horizon["iou"], "precision": horizon["precision"],
                              "recall": horizon["recall"],
                              "FP pixels": horizon["false_positive_pixels"],
                              "FN pixels": horizon["false_negative_pixels"]})
if scenarios:
    frame = pd.DataFrame(scenarios)
    display(frame.head(20))
    pivot = frame.pivot_table(index="simulation", columns="horizon_min", values="IoU", aggfunc="first")
    display(pivot.style.background_gradient(cmap="viridis").format(precision=3))
else:
    print("No per-scenario gallery metrics found. New eval jobs write gallery_metrics.json.")

## 8. Launching an instrumented CPU+GPU workflow

The notebook **does not schedule jobs automatically**. To create the pipeline, run the commands in `orchestration/observe/README.md` using your existing Fuzzball CLI:

1. Generate and upload a *fresh* GPU pilot package from this branch, so it includes training telemetry and the gallery exporter.
2. Package and upload the real ForeFire sources with `package_physics.py`.
3. Generate a Fuzzfile using `build_workflow.py` (two 12-core physics shards + one/two GPU training jobs).
4. Validate and submit with the Fuzzball CLI, then paste the real workflow ID into section 5.
5. Retrieve each `*-results.tgz` to unique directories; set `RUNS_DIR` and rerun the analysis cells.

**DDP:** `ml/train_arrival_ddp.py` supports real `torchrun` rendezvous on two reachable ranks, with explicit start/end events. The Fuzzball CPU/GPU overlap workflow does **not** automatically switch to DDP; that requires a verified cross-node rendezvous and its own job launch. No fake transition is displayed.

**Claims:** Dataset is simulator-generated; numbers represent agreement with ForeFire, not observed real-world fire accuracy. Never equate instantaneous telemetry snapshots with average utilization.